# Class-conditional diffusion model (DDPM) for synthetic defect images

A class-conditional **DDPM** built with 🤗 Diffusers generates 128×128 images of defective surfaces, one class per
defect family, to augment the scarce defect class of an industrial inspection dataset.

* **Model:** `UNet2DModel` (6 resolution levels, 128→512 channels, self-attention at 16×16) with
  `num_class_embeds=2` for class conditioning.
* **Training:** ε-prediction MSE loss, `DDPMScheduler` with 1000 steps, AdamW (lr 1e-4), cosine schedule with
  500 warm-up steps, fp16 mixed precision via 🤗 Accelerate, 100 epochs.
* **Sampling:** a manual class-conditional sampling loop (100 denoising steps) generates 500 images per class.
* **Downstream use:** the generated defect images are the "defect" training class of the classifier in
  `3_resnet18_trained_on_synthetic_defects.ipynb`, which is evaluated on *real* images only.

The training loop and data pipeline are adapted from the official 🤗 Diffusers training tutorial; the class
conditioning and the conditional sampling loop were added for this use case. This notebook was saved without outputs.

**Data:** proprietary defect images (not included), `imagefolder` layout with one sub-folder per class.


## Installing the dependencies

we use the [
Datasets](https://huggingface.co/docs/datasets/index) library to load and preprocess image datasets and the [Accelerate](https://huggingface.co/docs/accelerate/index) library to simplify training on any number of GPUs, with features like automatic gradient accumulation and tensorboard logging. Let's install them here:

In [ ]:
%pip install -q -U diffusers[training] datasets accelerate

In [ ]:
import os

# Root folder of the dataset(s) used in this notebook.
# Set the DATA_ROOT environment variable or edit the default below.
# On Google Colab you can first mount Drive with:
#   from google.colab import drive; drive.mount('/content/drive')
DATA_ROOT = os.environ.get("DATA_ROOT", "./data")

## Config

For convenience, we define a configuration grouping all the training hyperparameters. This would be similar to the arguments used for a [training script](https://github.com/huggingface/diffusers/tree/main/examples).
Here we choose reasonable defaults for hyperparameters like `num_epochs`, `learning_rate`, `lr_warmup_steps`, but feel free to adjust them if you train on your own dataset. For example, `num_epochs` can be increased to 100 for better visual quality.

In [ ]:
from dataclasses import dataclass

@dataclass
class TrainingConfig:
    image_size = 128  # the generated image resolution
    train_batch_size = 16
    eval_batch_size = 16  # how many images to sample during evaluation
    num_epochs = 100
    gradient_accumulation_steps = 1
    learning_rate = 1e-4
    lr_warmup_steps = 500
    save_image_epochs = 100
    save_model_epochs = 100
    mixed_precision = 'fp16'  # `no` for float32, `fp16` for automatic mixed precision
    output_dir = 'ddpm-defects-128'  # local output folder for the trained pipeline

    push_to_hub = False  # whether to upload the saved model to the HF Hub
    hub_private_repo = False
    overwrite_output_dir = True  # overwrite the old model when re-running the notebook
    seed = 0

config = TrainingConfig()

## Loading the dataset

We will use our image dataset.

In [ ]:
# Training images: DATA_ROOT/diffusion_defects/<class_name>/*.png (one folder per defect class)
train_data_dir = f"{DATA_ROOT}/diffusion_defects"

In [ ]:
from datasets import load_dataset


config.dataset_name = "imagefolder"
# Load the dataset, which returns a DatasetDict
raw_dataset = load_dataset(config.dataset_name, data_dir=train_data_dir)
# Extract the 'train' split and assign it to the dataset variable
dataset = raw_dataset['train']

🤗 Datasets offer a handy `set_transform()` method to apply the image transformations on the fly during training:

In [ ]:
from torchvision import transforms

preprocess = transforms.Compose(
    [
        transforms.Resize((config.image_size, config.image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),
    ]
)

def transform(examples):
    # Process images as before
    # Ensure you handle the grayscale conversion if needed, currently it converts to RGB
    # If your images are grayscale, you might want to keep them as 1 channel or convert to 3 channels
    # for the UNet model which expects 3 input channels (as defined previously).
    # If keeping grayscale, change in_channels in UNet and Normalize accordingly.
    images = [preprocess(image.convert("RGB")) for image in examples["image"]] # Or .convert("L") for grayscale

    # Extract labels from the examples
    # 'examples' is a dictionary where keys are column names ('image', 'label')
    # and values are lists of data for each example in the batch.
    labels = examples["label"]

    # Return a dictionary with both processed images and labels
    return {"images": images, "labels": labels}

dataset.set_transform(transform)

Let's see what they look like now

In [ ]:
import matplotlib.pyplot as plt
import torch # Import torch if not already imported

fig, axs = plt.subplots(1, 4, figsize=(16, 4))

displayed_classes = []
images_to_display = []
labels_to_display = []

# Iterate through the dataset to find images from different classes
for i in range(len(dataset)):
    sample = dataset[i]
    image = sample["images"]
    label = sample["labels"]

    # Check if the class label has already been displayed
    if label not in displayed_classes:
        images_to_display.append(image)
        labels_to_display.append(label)
        displayed_classes.append(label)

    # Stop once we have collected 4 images from different classes
    if len(images_to_display) == 4:
        break

# If there are fewer than 4 unique classes, display all collected images
num_images_to_show = len(images_to_display)

# Display the collected images
for i in range(num_images_to_show):
    axs[i].imshow(images_to_display[i].permute(1, 2, 0).numpy() / 2 + 0.5)
    axs[i].set_title(f"Class: {labels_to_display[i]}") # Add title to show class label
    axs[i].set_axis_off()

# Hide any unused subplots if fewer than 4 unique classes were found
for i in range(num_images_to_show, 4):
    fig.delaxes(axs[i])

fig.tight_layout() # Adjust layout to prevent titles overlapping
fig.show()

Now that all our images have the same size and are converted to tensors, we can create the dataloader we will use for training.

In [ ]:
import torch

train_dataloader = torch.utils.data.DataLoader(dataset, batch_size=config.train_batch_size, shuffle=True)

## Defining the diffusion model

Here we set up our diffusion model. Diffusion models are neural networks that are trained to predict slightly less noisy images from a noisy input. At inference, they can be used to iteratively transform a random noise to generate an image:

<p align="center">
    <img src="https://user-images.githubusercontent.com/10695622/174349667-04e9e485-793b-429a-affe-096e8199ad5b.png" width="800"/>
    <br>
    <em> Figure from DDPM paper (https://arxiv.org/abs/2006.11239). </em>
<p>

Don't worry too much about the math if you're not familiar with it, the import part to remember is that our model corresponds to the arrow $p_{\theta}(x_{t-1}|x_{t})$ (which is a fancy way of saying: predict a slightly less noisy image).

The interesting part is that it's really easy to add some noise to an image, so the training can happen in a semi-supervised fashion as follows:
1. Take an image from the training set.
2. Apply to it some random noise $t$ times (this will give the $x_{t-1}$ and the $x_{t}$ in the figure above).
3. Give this noisy image to the model along with the value of $t$.
4. Compute a loss from the output of the model and the noised image $x_{t-1}$.

Then we can apply gradient descent and repeat this process multiple times.

Most diffusion models use architectures that are some variant of a [U-net](https://arxiv.org/abs/1505.04597) and that's what we'll use here.

![](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/unet-model.png)

In a nutshell:
- the model has the input image go through several blocks of ResNet layers which halves the image size by 2
- then through the same number of blocks that upsample it again.
- there are skip connections linking the features on the downample path to the corresponsding layers in the upsample path.

A key feature of this model is that it predicts images of the same size as the input, which is exactly what we need here.

Diffusers provides us a handy `UNet2DModel` class which creates the desired architecture in PyTorch.

Let's create a U-net for our desired image size.
Note that `down_block_types` correspond to the downsampling blocks (green on the diagram above), and `up_block_types` are the upsampling blocks (red on the diagram):

In [ ]:
from diffusers import UNet2DModel

# Number of defect classes used for conditioning
num_wood_classes = 2

model = UNet2DModel(
    sample_size=config.image_size,
    in_channels=3,  # Still 3 if you convert grayscale to RGB or repeat channels
    out_channels=3,  # Output is still 3 channels (predicting noise for 3 channels)
    layers_per_block=2,
    block_out_channels=(128, 128, 256, 256, 512, 512),
    down_block_types=(
        "DownBlock2D",
        "DownBlock2D",
        "DownBlock2D",
        "DownBlock2D",
        "AttnDownBlock2D",
        "DownBlock2D",
    ),
    up_block_types=(
        "UpBlock2D",
        "AttnUpBlock2D",
        "UpBlock2D",
        "UpBlock2D",
        "UpBlock2D",
        "UpBlock2D"
      ),
    # class conditioning
    num_class_embeds=num_wood_classes
)

Let's get a sample image from our dataset and pass it into our model. We just need to add a batch dimension:

In [ ]:
sample_image = dataset[0]['images'].unsqueeze(0)
print('Input shape:', sample_image.shape)

Input shape: torch.Size([1, 3, 128, 128])


And let's check the output is a tensor of the same exact shape:

In [ ]:
print('Output shape:', model(sample_image, timestep=0, class_labels=torch.tensor([0])).sample.shape)

Output shape: torch.Size([1, 3, 128, 128])


Great!

Note that our model takes in the (noisy) image and also the current time-step (as we saw before in the training overview). That time-step information is converted for the model using a sinusoidal positional embedding, similar to what Transformer models often do.

Now that we have our model, we just need an object to *add noise to an image*. This is done by the **schedulers** in the `diffusers` library.

## Defining the noise scheduler

Depending on the diffusion algorithm you want to use, the way images are noised is slightly different. That's why 🤗 Diffusers contains different scheduler classes which each define the algorithm-specific diffusion steps. Here we are going to use the `DDPMScheduler` which corresponds to the training denoising and training algorithm proposed in [Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239).

In [ ]:
from diffusers import DDPMScheduler

noise_scheduler = DDPMScheduler(num_train_timesteps=1000)

Let's see how this noise scheduler works: it takes a batch of images from the trainng set (here we will reuse the batch of one image `sample_image` form before), a batch of random noise of the same shape and the timesteps for each image (which correspond to the number of times we want to apply noise to each image):

In [ ]:
import torch
from PIL import Image

noise = torch.randn(sample_image.shape)
timesteps = torch.LongTensor([50])
noisy_image = noise_scheduler.add_noise(sample_image, noise, timesteps)

Image.fromarray(((noisy_image.permute(0, 2, 3, 1) + 1.0) * 127.5).type(torch.uint8).numpy()[0])

<PIL.Image.Image image mode=RGB size=128x128>

In the DDPM algorithm, the training objective of the model is then to be able to predict the noise we used in `noise_scheduler.add_noise`, so the loss at this step would be:

In [ ]:
import torch.nn.functional as F

noise_pred = model(noisy_image, timesteps, class_labels=torch.tensor([0])).sample
loss = F.mse_loss(noise_pred, noise)

## Setting up training

We have all we need to be able to train our model! Let's use a standard AdamW optimizer:

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

 And a cosine learning rate schedule:

In [ ]:
from diffusers.optimization import get_cosine_schedule_with_warmup

lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=config.lr_warmup_steps,
    num_training_steps=(len(train_dataloader) * config.num_epochs),
)

To evaluate our model, we use the `DDPMPipeline` which is an easy way to perform end-to-end inference. We will use this pipeline to generate a batch of sample images and save it as a grid to the disk.

In [ ]:
from diffusers import DDPMPipeline
from PIL import Image
import math

def make_grid(images, rows, cols):
    w, h = images[0].size
    grid = Image.new('RGB', size=(cols*w, rows*h))
    for i, image in enumerate(images):
        grid.paste(image, box=(i%cols*w, i//cols*h))
    return grid

With this in end, we can group all together and write our training function. This just wraps the training step we saw in the previous section in a loop, using Accelerate for easy TensorBoard logging, gradient accumulation, mixed precision training and multi-GPUs or TPU training.

In [ ]:
from accelerate import Accelerator
from accelerate.utils import ProjectConfiguration
from huggingface_hub import create_repo, upload_folder

from tqdm.auto import tqdm
from pathlib import Path
import os

def train_loop(config, model, noise_scheduler, optimizer, train_dataloader, lr_scheduler):
    # Initialize accelerator and tensorboard logging
    logging_dir = os.path.join(config.output_dir, "logs")
    accelerator_project_config = ProjectConfiguration(project_dir=config.output_dir, logging_dir=logging_dir)
    accelerator = Accelerator(
        mixed_precision=config.mixed_precision,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        log_with="tensorboard",
        project_config=accelerator_project_config,
    )
    if accelerator.is_main_process:
        if config.push_to_hub:
            repo_id = create_repo(
                repo_id=Path(config.output_dir).name, exist_ok=True
            ).repo_id
        elif config.output_dir is not None:
            os.makedirs(config.output_dir, exist_ok=True)
        accelerator.init_trackers("train_example")

    # Prepare everything
    # There is no specific order to remember, you just need to unpack the
    # objects in the same order you gave them to the prepare method.
    model, optimizer, train_dataloader, lr_scheduler = accelerator.prepare(
        model, optimizer, train_dataloader, lr_scheduler
    )

    global_step = 0

    # Now you train the model
    for epoch in range(config.num_epochs):
        progress_bar = tqdm(total=len(train_dataloader), disable=not accelerator.is_local_main_process)
        progress_bar.set_description(f"Epoch {epoch}")

        for step, batch in enumerate(train_dataloader):
            clean_images = batch['images']
            labels = batch['labels']
            # Sample noise to add to the images
            noise = torch.randn(clean_images.shape).to(clean_images.device)
            bs = clean_images.shape[0]

            # Sample a random timestep for each image
            timesteps = torch.randint(0, noise_scheduler.num_train_timesteps, (bs,), device=clean_images.device).long()

            # Add noise to the clean images according to the noise magnitude at each timestep
            # (this is the forward diffusion process)
            noisy_images = noise_scheduler.add_noise(clean_images, noise, timesteps)

            with accelerator.accumulate(model):
                # Predict the noise residual
                noise_pred = model(noisy_images, timesteps, class_labels=labels, return_dict=False)[0]
                loss = F.mse_loss(noise_pred, noise)
                accelerator.backward(loss)

                accelerator.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()

            progress_bar.update(1)
            logs = {"loss": loss.detach().item(), "lr": lr_scheduler.get_last_lr()[0], "step": global_step}
            progress_bar.set_postfix(**logs)
            accelerator.log(logs, step=global_step)
            global_step += 1

        # After each epoch you optionally sample some demo images with evaluate() and save the model
        if accelerator.is_main_process:
            pipeline = DDPMPipeline(unet=accelerator.unwrap_model(model), scheduler=noise_scheduler)
            if (epoch + 1) % config.save_model_epochs == 0 or epoch == config.num_epochs - 1:
                if config.push_to_hub:
                    upload_folder(
                        repo_id=repo_id,
                        folder_path=config.output_dir,
                        commit_message=f"Epoch {epoch}",
                        ignore_patterns=["step_*", "epoch_*"],
                    )
                else:
                    pipeline.save_pretrained(config.output_dir)

## Let's train!

Let's launch the training (including multi-GPU training) from the notebook using Accelerate's `notebook_launcher` function:

In [ ]:
from accelerate import notebook_launcher
args = (config, model, noise_scheduler, optimizer, train_dataloader, lr_scheduler)

notebook_launcher(train_loop, args, num_processes=1)

Launching training on one GPU.


  deprecate("direct config name access", "1.0.0", deprecation_message, standard_warn=False)


In [ ]:
from diffusers import UNet2DModel, DDPMScheduler
import torch
from PIL import Image
import os
import glob
import math
from IPython.display import display
from tqdm.auto import tqdm # Import tqdm for progress bar


# Specify the directory where the trained model is saved
output_dir = 'ddpm-defects-128' # Main output directory
result_dir = 'ddpm-defects-128-samples'
# Define the specific paths to the saved UNet and Scheduler
unet_save_path = os.path.join(output_dir, 'unet') # Path where UNet config/weights are
scheduler_save_path = os.path.join(output_dir, 'scheduler') # Path where Scheduler config is

# Load the trained UNet model and Noise Scheduler directly
try:
    # Load the UNet model from its specific subdirectory
    model = UNet2DModel.from_pretrained(unet_save_path)
    print(f"UNet model loaded successfully from {unet_save_path}.")

    # Load the noise scheduler from its specific subdirectory
    noise_scheduler = DDPMScheduler.from_pretrained(scheduler_save_path)
    print(f"Noise scheduler loaded successfully from {scheduler_save_path}.")

except Exception as e:
    print(f"Error loading model or scheduler from specified paths.")
    print(f"Attempted loading from: UNet='{unet_save_path}', Scheduler='{scheduler_save_path}'")
    print(e)
    # Re-raise the exception to stop execution if loading fails
    raise e

# If you are using a GPU, move the model to the GPU
if torch.cuda.is_available():
    model.to("cuda")
    device = "cuda"
    print("Model moved to GPU.")
else:
    device = "cpu"
    model.to(device) # Ensure model is on CPU if no GPU
    print("Model running on CPU.")

# Define the number of images to generate per class
num_images_per_class = 500

# Define the number of inference steps (same as num_train_timesteps in scheduler)
num_inference_steps = noise_scheduler.config.num_train_timesteps # Use total timesteps for DDPM sampling
reduced_inference_steps = 100 # You can choose to use fewer steps for faster inference

# Get the number of classes from your model config
try:
    num_classes = model.config.num_class_embeds
    print(f"Detected {num_classes} classes from the model config.")
except AttributeError:
    print("Could not find num_class_embeds in model.config.")
    # If you know the number of classes, you can hardcode it here
    num_classes = 2 # Example: hardcode if config not found
    print(f"Assuming {num_classes} classes for generation.")

# Define a smaller batch size for inference to avoid OOM
inference_batch_size = 16 # You can adjust this based on your GPU memory

# Iterate through each class and generate images
for class_id in range(num_classes):
    print(f"Generating images for class: {class_id}")

    # Create the output folder for this class
    output_folder_class = os.path.join(result_dir, f"new_generated_images_class_{class_id}")
    os.makedirs(output_folder_class, exist_ok=True)

    generated_images = [] # List to store generated PIL images for this class

    # Generate images in smaller batches
    for i in tqdm(range(0, num_images_per_class, inference_batch_size), desc=f"Generating batches for class {class_id}"):
        batch_size = min(inference_batch_size, num_images_per_class - i) # Handle the last batch

        # Create a tensor of class labels for the batch, all set to the current class_id
        class_labels = torch.full(
            (batch_size,),           # Batch size for this iteration
            class_id,                          # The class ID for this batch
            device=device,                     # Move to the model's device
            dtype=torch.long                   # Use long data type for labels
        )

        # Generate initial random noise for the batch
        if 'config' not in locals():
             print("Error: 'config' object not found. Please run the config cell.")
             image_size = model.config.sample_size
             num_channels = model.config.in_channels
             if image_size is None or num_channels is None:
                 raise ValueError("Could not determine image size or channels. 'config' object is missing.")
        else:
             image_size = config.image_size
             num_channels = model.config.in_channels

        sample = torch.randn(batch_size, num_channels, image_size, image_size).to(device)

        # Set the scheduler's timesteps for inference
        noise_scheduler.set_timesteps(reduced_inference_steps)

        # Manual sampling loop for the current batch
        for t in noise_scheduler.timesteps:
            with torch.no_grad(): # Inference should be done without tracking gradients
                 timesteps = torch.full((batch_size,), t, device=device, dtype=torch.long)
                 noise_pred = model(sample, timesteps, class_labels=class_labels).sample

            sample = noise_scheduler.step(noise_pred, t, sample).prev_sample

        # Convert the generated tensors in this batch to PIL Images
        batch_generated_images = (sample / 2 + 0.5).clamp(0, 1)
        batch_generated_images = batch_generated_images.cpu().permute(0, 2, 3, 1).numpy()
        batch_generated_images = (batch_generated_images * 255).round().astype("uint8")
        batch_generated_images = [Image.fromarray(s) for s in batch_generated_images]

        # Append the generated images from this batch to the list
        generated_images.extend(batch_generated_images)

        # Save the generated images from this batch
        start_index = i
        end_index = i + batch_size
        for j in range(batch_size):
             image_index_in_total = start_index + j
             generated_images[image_index_in_total].save(f"{output_folder_class}/new_generated_image_class_{class_id}_{image_index_in_total:04d}.png")


    print(f"Successfully generated and saved {len(generated_images)} images for class {class_id}.")

print("Finished generating images for all classes.")

# Optional: Display a grid of images from the last generated class for review
if 'generated_images' in locals() and len(generated_images) > 0:
    print(f"\nDisplaying a grid of generated images for the last class ({class_id}):")
    # Ensure make_grid function is defined (it was in an earlier cell)
    # If not, define it here or ensure the previous cell ran.
    # Assuming make_grid and Image are available from previous imports

    # Select up to 16 images for the grid from the full list of generated images
    images_for_grid = generated_images[:16]
    grid_rows = min(4, math.ceil(len(images_for_grid) / 4))
    grid_cols = min(4, len(images_for_grid)) # Adjust if less than 4 images

    try:
        # Ensure images are PIL Images before passing to make_grid
        image_grid = make_grid(images_for_grid, rows=grid_rows, cols=grid_cols)
        display(image_grid)
    except Exception as e:
        print(f"Could not display image grid: {e}")
        print("Please ensure the 'make_grid' function and 'Image' are imported and available.")

UNet model loaded successfully from ddpm-defects-128/unet.
Noise scheduler loaded successfully from ddpm-defects-128/scheduler.
Model moved to GPU.
Detected 2 classes from the model config.
Generating images for class: 0


Successfully generated and saved 500 images for class 0.
Generating images for class: 1


Successfully generated and saved 500 images for class 1.
Finished generating images for all classes.

Displaying a grid of generated images for the last class (1):


<PIL.Image.Image image mode=RGB size=512x512>